In [5]:
# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# ==========================================
# 2. LOAD DATA
# ==========================================
df = pd.read_csv("traffic_dataset_25k.csv")

# ==========================================
# 3. PREPROCESSING
# ==========================================

# Convert Time → Hour
df['Hour'] = pd.to_datetime(df['Time'], format='mixed').dt.hour

# Encode categorical columns
le_day = LabelEncoder()
le_junction = LabelEncoder()
le_road = LabelEncoder()
le_weather = LabelEncoder()
le_target = LabelEncoder()

df['Day'] = le_day.fit_transform(df['Day'])
df['Junction'] = le_junction.fit_transform(df['Junction'])
df['RoadType'] = le_road.fit_transform(df['RoadType'])
df['Weather'] = le_weather.fit_transform(df['Weather'])
df['Traffic'] = le_target.fit_transform(df['Traffic'])

# Features & Target
X = df[['Hour','Day','Junction','RoadType','Weather',
        'CarCount','BikeCount','BusCount','TruckCount','IsHoliday']]
y = df['Traffic']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# ==========================================
# 4. DECISION TREE
# ==========================================
dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)

dt_acc = accuracy_score(y_test, dt_model.predict(X_test))
print("Decision Tree Accuracy:", dt_acc)

# ==========================================
# 5. ANN MODEL
# ==========================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ann_model = MLPClassifier(hidden_layer_sizes=(12,12), max_iter=300)
ann_model.fit(X_train_scaled, y_train)

ann_acc = accuracy_score(y_test, ann_model.predict(X_test_scaled))
print("ANN Accuracy:", ann_acc)

# ==========================================
# 6. CONTROL FUNCTION
# ==========================================
def control_signal(label):
    if label == "Low":
        return 20
    elif label == "Medium":
        return 40
    else:
        return 70

# ==========================================
# 7. USER INPUT
# ==========================================
def safe_transform(encoder, value):
    if value in encoder.classes_:
        return encoder.transform([value])[0]
    else:
        print(f"Warning: '{value}' not seen in training, using default.")
        return 0
print("\n--- ENTER INPUT ---")

hour = int(input("Hour (0-23): "))
day = input("Day: ")
junction = input("Junction (J1-J4): ")
road = input("Road Type (Highway/City/Residential): ")
weather = input("Weather (Sunny/Rainy/Cloudy): ")
holiday = int(input("Holiday (0/1): "))

car = int(input("Car count: "))
bike = int(input("Bike count: "))
bus = int(input("Bus count: "))
truck = int(input("Truck count: "))

# Encode input
input_data = pd.DataFrame([{
    'Hour': hour,
    'Day': safe_transform(le_day, day),
    'Junction': safe_transform(le_junction, junction),
    'RoadType': safe_transform(le_road, road),
    'Weather': safe_transform(le_weather, weather),
    'CarCount': car,
    'BikeCount': bike,
    'BusCount': bus,
    'TruckCount': truck,
    'IsHoliday': holiday
}])

# ==========================================
# 8. ENSEMBLE PREDICTION
# ==========================================

# Decision Tree
dt_pred = dt_model.predict(input_data)
dt_label = le_target.inverse_transform(dt_pred)[0]

# ANN
input_scaled = scaler.transform(input_data)
ann_pred = ann_model.predict(input_scaled)
ann_label = le_target.inverse_transform(ann_pred)[0]

# Combine intelligently
if dt_label == ann_label:
    final_label = dt_label
else:
    final_label = ann_label if ann_acc > dt_acc else dt_label

# ==========================================
# 9. FINAL OUTPUT
# ==========================================
green_time = control_signal(final_label)

print("\n--- FINAL OUTPUT ---")
print("Traffic Level:", final_label)
print("Green Signal Time:", green_time, "seconds")

Decision Tree Accuracy: 0.9766
ANN Accuracy: 0.999

--- ENTER INPUT ---

--- FINAL OUTPUT ---
Traffic Level: Medium
Green Signal Time: 40 seconds


In [8]:
# ==========================================
# TRAFFIC CONTROL SYSTEM WITH WEATHER INPUT
# Input: Hour, Day, IsHoliday, Weather (Sunny/Rainy/Cloudy)
# Output: For each junction -> predicted Car/Bike/Bus/Truck counts,
#         weighted load, and green signal time (seconds)
# ==========================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# ==========================================
# 1. LOAD AND PREPROCESS DATA
# ==========================================
df = pd.read_csv("traffic_dataset_25k.csv")

# Extract Hour from Time column
df['Hour'] = pd.to_datetime(df['Time'], format='mixed').dt.hour

# Encode categorical columns: Day, Junction, Weather
le_day = LabelEncoder()
le_junction = LabelEncoder()
le_weather = LabelEncoder()

df['Day'] = le_day.fit_transform(df['Day'])
df['Junction'] = le_junction.fit_transform(df['Junction'])
df['Weather'] = le_weather.fit_transform(df['Weather'])

# Features: Hour, Day, IsHoliday, Junction, Weather
X = df[['Hour', 'Day', 'IsHoliday', 'Junction', 'Weather']]

# Targets: separate counts for each vehicle type
y_car = df['CarCount']
y_bike = df['BikeCount']
y_bus = df['BusCount']
y_truck = df['TruckCount']

# Split into train/test
X_train, X_test, y_car_train, y_car_test = train_test_split(X, y_car, test_size=0.2, random_state=42)
_, _, y_bike_train, y_bike_test = train_test_split(X, y_bike, test_size=0.2, random_state=42)
_, _, y_bus_train, y_bus_test = train_test_split(X, y_bus, test_size=0.2, random_state=42)
_, _, y_truck_train, y_truck_test = train_test_split(X, y_truck, test_size=0.2, random_state=42)

# ==========================================
# 2. TRAIN REGRESSORS FOR EACH VEHICLE TYPE
# ==========================================
print("Training models for each vehicle type (with weather)...")
reg_car = RandomForestRegressor(n_estimators=100, min_samples_split=5, random_state=42)
reg_bike = RandomForestRegressor(n_estimators=100, min_samples_split=5, random_state=42)
reg_bus = RandomForestRegressor(n_estimators=100, min_samples_split=5, random_state=42)
reg_truck = RandomForestRegressor(n_estimators=100, min_samples_split=5, random_state=42)

reg_car.fit(X_train, y_car_train)
reg_bike.fit(X_train, y_bike_train)
reg_bus.fit(X_train, y_bus_train)
reg_truck.fit(X_train, y_truck_train)
import joblib

# Save label encoders
joblib.dump(le_day, "le_day.pkl")
joblib.dump(le_junction, "le_junction.pkl")
joblib.dump(le_weather, "le_weather.pkl")

# Save regressors
joblib.dump(reg_car, "reg_car.pkl")
joblib.dump(reg_bike, "reg_bike.pkl")
joblib.dump(reg_bus, "reg_bus.pkl")
joblib.dump(reg_truck, "reg_truck.pkl")

# Also save max_load (95th percentile from training)
max_load = np.percentile(train_weighted, 95)
joblib.dump(max_load, "max_load.pkl")

# Evaluate
def evaluate(reg, y_test, name):
    pred = reg.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    print(f"{name} MAE: {mae:.2f} vehicles")
    
evaluate(reg_car, y_car_test, "Car")
evaluate(reg_bike, y_bike_test, "Bike")
evaluate(reg_bus, y_bus_test, "Bus")
evaluate(reg_truck, y_truck_test, "Truck")
print()

# ==========================================
# 3. WEIGHTED LOAD FUNCTION (Passenger Car Equivalents)
# ==========================================
def weighted_load(car, bike, bus, truck):
    return car * 1.0 + bike * 0.5 + bus * 3.0 + truck * 2.5

# Determine typical max weighted load from training data (95th percentile)
train_weighted = weighted_load(y_car_train, y_bike_train, y_bus_train, y_truck_train)
max_load = np.percentile(train_weighted, 95)

def green_time_from_weighted_load(load, max_load=max_load):
    green = 15 + (load / max_load) * 75
    return int(np.clip(green, 15, 90))

# ==========================================
# 4. HELPER: SAFE TRANSFORM FOR UNSEEN CATEGORIES
# ==========================================
def safe_transform(encoder, value, default=0):
    if value in encoder.classes_:
        return encoder.transform([value])[0]
    else:
        print(f"Warning: '{value}' not seen in training. Using default ({default}).")
        return default

# ==========================================
# 5. PREDICTION FOR ALL JUNCTIONS (WITH WEATHER)
# ==========================================
def predict_all_junctions(hour, day_name, is_holiday, weather_str):
    """
    Given hour (0-23), day name, holiday flag, and weather (Sunny/Rainy/Cloudy),
    return a dictionary with junction names as keys containing predicted counts,
    weighted load, and green time.
    """
    # Encode day and weather
    day_encoded = safe_transform(le_day, day_name)
    weather_encoded = safe_transform(le_weather, weather_str)

    results = {}
    for junction_name in le_junction.classes_:
        junction_encoded = le_junction.transform([junction_name])[0]
        input_df = pd.DataFrame([{
            'Hour': hour,
            'Day': day_encoded,
            'IsHoliday': is_holiday,
            'Junction': junction_encoded,
            'Weather': weather_encoded
        }])
        
        # Predict per type
        car = max(0, int(reg_car.predict(input_df)[0]))
        bike = max(0, int(reg_bike.predict(input_df)[0]))
        bus = max(0, int(reg_bus.predict(input_df)[0]))
        truck = max(0, int(reg_truck.predict(input_df)[0]))
        
        load = weighted_load(car, bike, bus, truck)
        green = green_time_from_weighted_load(load)
        
        results[junction_name] = {
            'car': car, 'bike': bike, 'bus': bus, 'truck': truck,
            'weighted_load': round(load, 1),
            'green_time_sec': green
        }
    return results

# ==========================================
# 6. USER INPUT (time, holiday, AND weather)
# ==========================================
print("=== TRAFFIC CONTROL SYSTEM (with Weather) ===")
print("Enter the following information (no manual vehicle counts):")
hour = int(input("Hour (0-23): "))
day = input("Day of week (e.g., Monday, Tuesday, ...): ")
holiday = int(input("Is holiday? (1 = Yes, 0 = No): "))
weather = input("Weather (Sunny / Rainy / Cloudy): ").strip().capitalize()

control = predict_all_junctions(hour, day, holiday, weather)

# ==========================================
# 7. DISPLAY RESULTS
# ==========================================
print("\n--- RECOMMENDED GREEN SIGNAL TIMES (for main movement at each junction) ---")
print("(These times assume a two-phase signal; actual controller cycles through phases.)\n")
for junction, info in control.items():
    print(f"{junction}:")
    print(f"  Predicted: {info['car']} cars, {info['bike']} bikes, {info['bus']} buses, {info['truck']} trucks")
    print(f"  Weighted load = {info['weighted_load']} PCE")
    print(f"  → Green time = {info['green_time_sec']} seconds\n")

total_weighted = sum(info['weighted_load'] for info in control.values())
print(f"Total weighted load across all junctions: {total_weighted:.1f} PCE")
print("Note: Each junction operates independently. Green times shown are for the busiest approach.")

Training models for each vehicle type (with weather)...
Car MAE: 15.43 vehicles
Bike MAE: 7.74 vehicles
Bus MAE: 2.12 vehicles
Truck MAE: 2.11 vehicles

=== TRAFFIC CONTROL SYSTEM (with Weather) ===
Enter the following information (no manual vehicle counts):

--- RECOMMENDED GREEN SIGNAL TIMES (for main movement at each junction) ---
(These times assume a two-phase signal; actual controller cycles through phases.)

J1:
  Predicted: 119 cars, 56 bikes, 10 buses, 12 trucks
  Weighted load = 207.0 PCE
  → Green time = 74 seconds

J2:
  Predicted: 111 cars, 55 bikes, 10 buses, 11 trucks
  Weighted load = 196.0 PCE
  → Green time = 71 seconds

J3:
  Predicted: 116 cars, 55 bikes, 11 buses, 11 trucks
  Weighted load = 204.0 PCE
  → Green time = 73 seconds

J4:
  Predicted: 118 cars, 58 bikes, 12 buses, 11 trucks
  Weighted load = 210.5 PCE
  → Green time = 75 seconds

Total weighted load across all junctions: 817.5 PCE
Note: Each junction operates independently. Green times shown are for the

In [7]:
import pandas as pd
df = pd.read_csv("Metro_Interstate_Traffic_Volume.csv")
df.to_csv("Metro_Interstate_Traffic_Volume.csv", index=False, encoding="utf-8")

In [8]:
#plot traffic
# ==============================================
# TRAINING SCRIPT FOR TRAFFIC CONTROL SYSTEM
# Trains separate RandomForest models for car, bike, bus, truck
# Saves models, label encoders, and max weighted load (95th percentile)
# ==============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import joblib

# ------------------------------
# 1. Load and preprocess data
# ------------------------------
df = pd.read_csv("traffic_dataset_25k.csv")   # adjust filename if needed

# Convert Time column to hour (assuming format like "04:00:00")
df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.hour

# Encode categorical columns
le_day = LabelEncoder()
le_junction = LabelEncoder()
le_weather = LabelEncoder()

df['Day'] = le_day.fit_transform(df['Day'])
df['Junction'] = le_junction.fit_transform(df['Junction'])
df['Weather'] = le_weather.fit_transform(df['Weather'])

# Features
X = df[['Hour', 'Day', 'IsHoliday', 'Junction', 'Weather']]
y_car = df['CarCount']
y_bike = df['BikeCount']
y_bus = df['BusCount']
y_truck = df['TruckCount']

# ------------------------------
# 2. Time‑based split (no data leakage)
# ------------------------------
# Sort by original order (assuming dataframe is chronological)
# If not sorted, sort by a datetime column – here we assume row order is temporal.
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_car_train, y_car_test = y_car.iloc[:split_idx], y_car.iloc[split_idx:]
y_bike_train, y_bike_test = y_bike.iloc[:split_idx], y_bike.iloc[split_idx:]
y_bus_train, y_bus_test = y_bus.iloc[:split_idx], y_bus.iloc[split_idx:]
y_truck_train, y_truck_test = y_truck.iloc[:split_idx], y_truck.iloc[split_idx:]

# ------------------------------
# 3. Train models
# ------------------------------
print("Training RandomForest regressors...")
reg_car = RandomForestRegressor(n_estimators=100, min_samples_split=5, random_state=42)
reg_bike = RandomForestRegressor(n_estimators=100, min_samples_split=5, random_state=42)
reg_bus = RandomForestRegressor(n_estimators=100, min_samples_split=5, random_state=42)
reg_truck = RandomForestRegressor(n_estimators=100, min_samples_split=5, random_state=42)

reg_car.fit(X_train, y_car_train)
reg_bike.fit(X_train, y_bike_train)
reg_bus.fit(X_train, y_bus_train)
reg_truck.fit(X_train, y_truck_train)

# ------------------------------
# 4. Save models and encoders
# ------------------------------
joblib.dump(le_day, "le_day.pkl")
joblib.dump(le_junction, "le_junction.pkl")
joblib.dump(le_weather, "le_weather.pkl")
joblib.dump(reg_car, "reg_car.pkl")
joblib.dump(reg_bike, "reg_bike.pkl")
joblib.dump(reg_bus, "reg_bus.pkl")
joblib.dump(reg_truck, "reg_truck.pkl")

# ------------------------------
# 5. Compute max weighted load (95th percentile from training)
# ------------------------------
def weighted_load(car, bike, bus, truck):
    return car * 1.0 + bike * 0.5 + bus * 3.0 + truck * 2.5

train_load = weighted_load(y_car_train, y_bike_train, y_bus_train, y_truck_train)
max_load = np.percentile(train_load, 95)
joblib.dump(max_load, "max_load.pkl")
print(f"Max weighted load (95th percentile) = {max_load:.1f}")

# ------------------------------
# 6. Evaluate on test set
# ------------------------------
def evaluate(reg, y_true, name):
    pred = reg.predict(X_test)
    mae = mean_absolute_error(y_true, pred)
    print(f"{name} MAE: {mae:.2f}")

evaluate(reg_car, y_car_test, "Car")
evaluate(reg_bike, y_bike_test, "Bike")
evaluate(reg_bus, y_bus_test, "Bus")
evaluate(reg_truck, y_truck_test, "Truck")

print("Training completed and models saved.")

Training RandomForest regressors...
Max weighted load (95th percentile) = 260.0
Car MAE: 15.47
Bike MAE: 7.82
Bus MAE: 2.12
Truck MAE: 2.10
Training completed and models saved.


In [9]:
# ==============================================
# app.py – TRAINING SCRIPT
# ==============================================
##streamlit3
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error
import joblib

# ------------------------------
# Load dataset
# ------------------------------
df = pd.read_csv("traffic_dataset_25k.csv")

# Convert time → hour
df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.hour

# Encode categorical features
le_day = LabelEncoder()
le_junction = LabelEncoder()
le_weather = LabelEncoder()

df['Day'] = le_day.fit_transform(df['Day'])
df['Junction'] = le_junction.fit_transform(df['Junction'])
df['Weather'] = le_weather.fit_transform(df['Weather'])

# Features
X = df[['Hour', 'Day', 'IsHoliday', 'Junction', 'Weather']]

# Apply PCA
pca = PCA(n_components=3)
X = pca.fit_transform(X)

y_car = df['CarCount']
y_bike = df['BikeCount']
y_bus = df['BusCount']
y_truck = df['TruckCount']

# Time-based split
split_idx = int(len(df) * 0.8)

X_train, X_test = X[:split_idx], X[split_idx:]
y_car_train, y_car_test = y_car[:split_idx], y_car[split_idx:]
y_bike_train, y_bike_test = y_bike[:split_idx], y_bike[split_idx:]
y_bus_train, y_bus_test = y_bus[:split_idx], y_bus[split_idx:]
y_truck_train, y_truck_test = y_truck[:split_idx], y_truck[split_idx:]

# Train models
reg_car = RandomForestRegressor(n_estimators=100)
reg_bike = RandomForestRegressor(n_estimators=100)
reg_bus = RandomForestRegressor(n_estimators=100)
reg_truck = RandomForestRegressor(n_estimators=100)

reg_car.fit(X_train, y_car_train)
reg_bike.fit(X_train, y_bike_train)
reg_bus.fit(X_train, y_bus_train)
reg_truck.fit(X_train, y_truck_train)

# Save everything
joblib.dump(reg_car, "reg_car.pkl")
joblib.dump(reg_bike, "reg_bike.pkl")
joblib.dump(reg_bus, "reg_bus.pkl")
joblib.dump(reg_truck, "reg_truck.pkl")

joblib.dump(le_day, "le_day.pkl")
joblib.dump(le_junction, "le_junction.pkl")
joblib.dump(le_weather, "le_weather.pkl")
joblib.dump(pca, "pca.pkl")

print("✅ Models trained and saved")

✅ Models trained and saved
